# Protein-Ligand Binding Affinity (pKd) Prediction
**Target:** `neg_log10_affinity_M`  
**Data:** AlphaFold3 + HADDOCK3 docking outputs  
**Models:** Random Forest · XGBoost · 1D-CNN (hybrid)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import joblib

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Load Data

In [ ]:
df = pd.read_csv('combined_40k.csv')
print(f'Shape: {df.shape}')

## 3. iptm Filtering
**Add your `iptm` column here, then filter to iptm ≥ 0.7.**

In [ ]:
# --- ADD IPTM COLUMN HERE ---
# e.g. df = df.merge(iptm_df[['Job_ID', 'iptm']], on='Job_ID', how='left')

# Filter rows with iptm >= 0.7
# df = df[df['iptm'] >= 0.7].reset_index(drop=True)
# print(f'Shape after iptm filter: {df.shape}')

## 4. Preprocessing

In [ ]:
TARGET = 'neg_log10_affinity_M'

SCALAR_FEATURES = [
    'molecular_weight', 'logP', 'num_h_donors', 'num_h_acceptors',
    'num_rotatable_bonds', 'tpsa', 'num_rings', 'num_aromatic_rings',
    'num_atoms', 'num_heavy_atoms', 'formal_charge',
    'score', 'total', 'vdw', 'elec', 'desolv', 'bsa', 'num_interacting_residues'
]
MORGAN_FEATURES = [c for c in df.columns if c.startswith('morgan_fp_')]
ALL_FEATURES = SCALAR_FEATURES + MORGAN_FEATURES
n_scalar = len(SCALAR_FEATURES)

print(f'Scalar features : {len(SCALAR_FEATURES)}')
print(f'Morgan FP bits  : {len(MORGAN_FEATURES)}')

X = df[ALL_FEATURES].copy()
y = df[TARGET].values

# Impute missing HADDOCK values with column median
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X_df = pd.DataFrame(X_imputed, columns=ALL_FEATURES)

print(f'Missing after imputation: {X_df.isnull().sum().sum()}')

In [ ]:
# 70 / 15 / 15 split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_df.values, y, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15/0.85, random_state=SEED)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

# Scale scalar features only (Morgan FP bits are binary)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[:, :n_scalar] = scaler.fit_transform(X_train[:, :n_scalar])
X_val_scaled[:,   :n_scalar] = scaler.transform(X_val[:,   :n_scalar])
X_test_scaled[:,  :n_scalar] = scaler.transform(X_test[:,  :n_scalar])

In [ ]:
from scipy import stats

# Normality test on target (D'Agostino-Pearson; Shapiro-Wilk is limited to n<=5000)
stat, p_normal = stats.normaltest(y_train)
print(f'D\u2019Agostino-Pearson normality test on y_train: stat={stat:.4f}, p={p_normal:.4e}')
if p_normal > 0.05:
    CORR_METHOD = 'pearson'
    print('p > 0.05 → pKd is approximately normal → using Pearson correlation')
else:
    CORR_METHOD = 'spearman'
    print('p ≤ 0.05 → pKd is not normal → using Spearman correlation')

def evaluate(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    if CORR_METHOD == 'pearson':
        corr, corr_p = stats.pearsonr(y_true, y_pred)
    else:
        corr, corr_p = stats.spearmanr(y_true, y_pred)
    print(f'{name:30s}  R²={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}  '
          f'{CORR_METHOD.capitalize()} r={corr:.4f} (p={corr_p:.2e})')
    return {'model': name, 'R2': r2, 'RMSE': rmse, 'MAE': mae,
            'corr': corr, 'corr_p': corr_p, 'y_true': y_true, 'y_pred': y_pred}

results = {}

## 5. Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_features='sqrt',   # recommended for high-dim FP data
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=SEED
)
rf.fit(X_train, y_train)   # RF doesn't need scaled input
results['RF'] = evaluate('Random Forest', y_test, rf.predict(X_test))

In [ ]:
# Top-20 feature importances (MDI)
importances = pd.Series(rf.feature_importances_, index=ALL_FEATURES)
top20_rf = importances.nlargest(20)

plt.figure(figsize=(8, 5))
top20_rf.sort_values().plot(kind='barh', color='steelblue')
plt.title('Random Forest — Top 20 Feature Importances (MDI)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()
print(top20_rf)

## 6. XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.3,  # subsampling columns is important over 2000+ FP features
    min_child_weight=5,
    reg_alpha=0.1,         # L1 sparsity over FP bits
    reg_lambda=1.0,
    tree_method='hist',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    random_state=SEED,
    early_stopping_rounds=50,
    eval_metric='rmse'
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)
results['XGB'] = evaluate('XGBoost', y_test, xgb_model.predict(X_test))

In [ ]:
# Top-20 feature importances (gain)
xgb_imp = pd.Series(
    xgb_model.get_booster().get_score(importance_type='gain')
).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
xgb_imp.head(20).sort_values().plot(kind='barh', color='darkorange')
plt.title('XGBoost — Top 20 Feature Importances (Gain)')
plt.xlabel('Gain')
plt.tight_layout()
plt.show()
print(xgb_imp.head(20))

## 7. 1D-CNN (Hybrid)
- **CNN branch**: reads 2048 Morgan FP bits as a 1D signal → conv layers → global avg pool  
- **MLP branch**: processes the 18 scalar features  
- **Fusion**: concatenate → dense head → pKd

In [ ]:
class HybridCNN(nn.Module):
    def __init__(self, n_scalar, n_fp, dropout=0.3):
        super().__init__()
        self.fp_branch = nn.Sequential(
            nn.Conv1d(1, 64,  kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),                        # 2048 → 1024
            nn.Conv1d(64, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),                        # 1024 → 512
            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),                # → 256
            nn.Flatten()
        )
        self.scalar_branch = nn.Sequential(
            nn.Linear(n_scalar, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x_scalar, x_fp):
        fp_out = self.fp_branch(x_fp.unsqueeze(1))  # add channel dim
        sc_out = self.scalar_branch(x_scalar)
        return self.head(torch.cat([fp_out, sc_out], dim=1)).squeeze(1)

In [ ]:
def make_tensors(X_scaled, y_arr):
    return (
        torch.tensor(X_scaled[:, :n_scalar], dtype=torch.float32),
        torch.tensor(X_scaled[:, n_scalar:], dtype=torch.float32),
        torch.tensor(y_arr, dtype=torch.float32)
    )

BATCH = 256
train_loader = DataLoader(TensorDataset(*make_tensors(X_train_scaled, y_train)), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(*make_tensors(X_val_scaled,   y_val)),   batch_size=BATCH)
test_loader  = DataLoader(TensorDataset(*make_tensors(X_test_scaled,  y_test)),  batch_size=BATCH)

In [ ]:
cnn = HybridCNN(n_scalar=n_scalar, n_fp=len(MORGAN_FEATURES)).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in cnn.parameters() if p.requires_grad):,}')

optimizer = optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)
criterion = nn.MSELoss()

EPOCHS   = 100
PATIENCE = 15
best_val_loss = float('inf')
patience_counter = 0
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    cnn.train()
    tr_loss = 0
    for x_sc, x_fp, y_b in train_loader:
        x_sc, x_fp, y_b = x_sc.to(DEVICE), x_fp.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(cnn(x_sc, x_fp), y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cnn.parameters(), 1.0)
        optimizer.step()
        tr_loss += loss.item() * len(y_b)
    tr_loss /= len(y_train)

    cnn.eval()
    vl_loss = 0
    with torch.no_grad():
        for x_sc, x_fp, y_b in val_loader:
            x_sc, x_fp, y_b = x_sc.to(DEVICE), x_fp.to(DEVICE), y_b.to(DEVICE)
            vl_loss += criterion(cnn(x_sc, x_fp), y_b).item() * len(y_b)
    vl_loss /= len(y_val)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    scheduler.step()

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_counter = 0
        torch.save(cnn.state_dict(), 'cnn_best.pt')
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | Train RMSE: {np.sqrt(tr_loss):.4f} | Val RMSE: {np.sqrt(vl_loss):.4f}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

plt.figure(figsize=(8, 4))
plt.plot(np.sqrt(train_losses), label='Train RMSE')
plt.plot(np.sqrt(val_losses),   label='Val RMSE')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.title('CNN Training Curve')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
cnn.load_state_dict(torch.load('cnn_best.pt', map_location=DEVICE))
cnn.eval()
preds_cnn = []
with torch.no_grad():
    for x_sc, x_fp, _ in test_loader:
        preds_cnn.extend(cnn(x_sc.to(DEVICE), x_fp.to(DEVICE)).cpu().numpy())

results['CNN'] = evaluate('1D-CNN (Hybrid)', y_test, np.array(preds_cnn))

## 8. Model Comparison

In [ ]:
corr_label = f'{CORR_METHOD.capitalize()} r'
summary = pd.DataFrame([
    {'Model': 'Random Forest', 'R²': results['RF']['R2'],  'RMSE': results['RF']['RMSE'],  'MAE': results['RF']['MAE'],  corr_label: results['RF']['corr']},
    {'Model': 'XGBoost',       'R²': results['XGB']['R2'], 'RMSE': results['XGB']['RMSE'], 'MAE': results['XGB']['MAE'], corr_label: results['XGB']['corr']},
    {'Model': '1D-CNN',        'R²': results['CNN']['R2'], 'RMSE': results['CNN']['RMSE'], 'MAE': results['CNN']['MAE'], corr_label: results['CNN']['corr']},
]).set_index('Model')

print('=== Test Set Performance ===')
print(summary.to_string(float_format='{:.4f}'.format))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (key, name, color) in zip(axes, [('RF','Random Forest','steelblue'),
                                          ('XGB','XGBoost','darkorange'),
                                          ('CNN','1D-CNN','mediumseagreen')]):
    y_t, y_p = results[key]['y_true'], results[key]['y_pred']
    ax.scatter(y_t, y_p, alpha=0.2, s=5, color=color)
    lims = [min(y_t.min(), y_p.min()), max(y_t.max(), y_p.max())]
    ax.plot(lims, lims, 'k--', linewidth=1)
    ax.set_xlabel('True pKd')
    ax.set_ylabel('Predicted pKd')
    ax.set_title(f"{name}\nR²={results[key]['R2']:.3f}  RMSE={results[key]['RMSE']:.3f}")
plt.suptitle('Predicted vs True pKd — Test Set', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. Save Models

In [ ]:
joblib.dump(rf,        'rf_model.pkl')
joblib.dump(xgb_model, 'xgb_model.pkl')
joblib.dump(scaler,    'scaler.pkl')
joblib.dump(imputer,   'imputer.pkl')
# CNN already saved as cnn_best.pt
# Reload with:
#   model = HybridCNN(n_scalar, len(MORGAN_FEATURES))
#   model.load_state_dict(torch.load('cnn_best.pt'))
print('Saved: rf_model.pkl, xgb_model.pkl, cnn_best.pt, scaler.pkl, imputer.pkl')